In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 - Entrenamiento de modelos y MLflow
# MAGIC
# MAGIC **Control:** control-2
# MAGIC
# MAGIC **Experimento MLflow:** `control-2`
# MAGIC
# MAGIC Se entrenan y comparan 5 variantes:
# MAGIC
# MAGIC 1. `modelo-run-1` — Logistic Regression
# MAGIC 2. `modelo-run-2` — Decision Tree
# MAGIC 3. `modelo-run-3` — Random Forest
# MAGIC 4. `modelo-run-4` — Gradient Boosting
# MAGIC 5. `modelo-run-5` — Extra Trees
# MAGIC
# MAGIC Variable objetivo:
# MAGIC `supera_50k`

# COMMAND ----------

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Configuración

# COMMAND ----------

FEATURE_TABLE = "workspace.control2.control2_clientes_features"

EXPERIMENT_NAME = "control-2"

# Ruta del experimento. Si tu profesor exige que sea exactamente "control-2",
# este nombre es el que debes visualizar en MLflow.
EXPERIMENT_PATH = "/Shared/control-2"

mlflow.set_experiment(EXPERIMENT_PATH)

print("Experimento MLflow:", EXPERIMENT_PATH)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Cargar features

# COMMAND ----------

df_spark = spark.table(FEATURE_TABLE)
df = df_spark.toPandas()

print("Registros:", len(df))
print("Columnas:", len(df.columns))

display(df.head(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Separar variables predictoras y objetivo

# COMMAND ----------

TARGET = "supera_50k"
ID_COL = "cliente_id"

X = df.drop(columns=[TARGET, ID_COL])
y = df[TARGET].astype(int)

categorical_cols = [
    c for c in [
        "nivel_educacion",
        "sector_economico",
        "tipo_empleo",
        "ciudad",
        "segmento_ingreso"
    ]
    if c in X.columns
]

numeric_cols = [
    c for c in X.columns
    if c not in categorical_cols
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Porcentaje clase 1:", round(y.mean() * 100, 2), "%")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Definir las 5 variantes

# COMMAND ----------

models = {
    "modelo-run-1": LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42
    ),

    "modelo-run-2": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        random_state=42
    ),

    "modelo-run-3": RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ),

    "modelo-run-4": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "modelo-run-5": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Entrenamiento + registro en MLflow

# COMMAND ----------

results = []

for run_name, model in models.items():

    with mlflow.start_run(run_name=run_name) as run:

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model)
            ]
        )

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test, y_pred, zero_division=0
        )
        recall = recall_score(
            y_test, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_test, y_pred, zero_division=0
        )
        auc = roc_auc_score(y_test, y_prob)

        cm = confusion_matrix(y_test, y_pred)

        # Parámetros
        mlflow.log_param("run_name", run_name)
        mlflow.log_param("modelo", model.__class__.__name__)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("num_features", len(numeric_cols))
        mlflow.log_param("cat_features", len(categorical_cols))

        # Métricas
        mlflow.log_metric("accuracy", float(accuracy))
        mlflow.log_metric("precision", float(precision))
        mlflow.log_metric("recall", float(recall))
        mlflow.log_metric("f1_score", float(f1))
        mlflow.log_metric("roc_auc", float(auc))

        # Matriz de confusión
        cm_file = f"/tmp/{run_name}_confusion_matrix.txt"

        with open(cm_file, "w") as f:
            f.write("Matriz de confusión\n")
            f.write(str(cm))

        mlflow.log_artifact(cm_file)

        from mlflow.models import infer_signature

        # Generar la firma del modelo
        signature = infer_signature(
            X_train,
            pipeline.predict(X_train)
        )

        # Registrar modelo con signature
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
            signature=signature,
            input_example=X_train.iloc[[0]]
        )

        results.append({
            "run_name": run_name,
            "modelo": model.__class__.__name__,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": auc,
            "run_id": run.info.run_id
        })

        print(
            f"{run_name} | "
            f"{model.__class__.__name__} | "
            f"Accuracy={accuracy:.4f} | "
            f"Precision={precision:.4f} | "
            f"Recall={recall:.4f} | "
            f"F1={f1:.4f} | "
            f"AUC={auc:.4f}"
        )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Comparación de las 5 ejecuciones

# COMMAND ----------

results_df = pd.DataFrame(results).sort_values(
    by="roc_auc",
    ascending=False
)

display(results_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Mejor modelo

# COMMAND ----------

best = results_df.iloc[0]

print("======================================")
print("        MEJOR MODELO - CONTROL 2")
print("======================================")
print("Run:", best["run_name"])
print("Modelo:", best["modelo"])
print("Accuracy:", round(best["accuracy"], 4))
print("Precision:", round(best["precision"], 4))
print("Recall:", round(best["recall"], 4))
print("F1:", round(best["f1_score"], 4))
print("ROC-AUC:", round(best["roc_auc"], 4))
print("Run ID:", best["run_id"])
print("======================================")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Evidencia para entregar
# MAGIC
# MAGIC En la interfaz de MLflow abre el experimento **control-2** y toma un
# MAGIC screenshot donde sean visibles como mínimo las 5 ejecuciones:
# MAGIC
# MAGIC - modelo-run-1
# MAGIC - modelo-run-2
# MAGIC - modelo-run-3
# MAGIC - modelo-run-4
# MAGIC - modelo-run-5
# MAGIC
# MAGIC junto con sus métricas principales.
